# analyzing CGWB raw data

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
import pandas as pd

# 1. Reading the raw CGWB data

In [ ]:
cgwb=pd.read_csv(f'{WORKSPACE_ROOT}/1. Analyzing CGWB data/input/CGWB_Time_Series_GW - CGWB_Time_Series_GW.csv')

In [ ]:
cgwb

In [ ]:
for col in cgwb.columns:
  print(col)




#**2. Selecting only values from 2001**

In [ ]:
columns_to_keep = ['objectid', 'pointInRegion', 'station_id', 'station_name', '.geo'] + \
                  [col for col in cgwb.columns if col.startswith('2001') or (col.startswith('20') and not col.startswith('2000'))]

In [ ]:
cgwb_2001_onwards = cgwb[columns_to_keep]

In [ ]:
cgwb_2001_onwards

#3. columns mentioning the season number are renamed to mentioned the months each season represent


In [ ]:
# Define the mapping dictionary for column renaming
column_mapping = {
    'objectid': 'objectid',
    'pointInRegion': 'pointInRegion',
    'station_id': 'station_id',
    'station_name': 'station_name',
    '.geo': '.geo'
}

# Rename the columns representing seasons
for year in range(2001, 2024):
    column_mapping[f'{year}_season1'] = f'{year}_Jan_March'
    column_mapping[f'{year}_season2'] = f'{year}_April_June'
    column_mapping[f'{year}_season3'] = f'{year}_Jul_Sept'
    column_mapping[f'{year}_season4'] = f'{year}_Oct_Dec'

# Apply column renaming
cgwb_2001_onwards_renamed = cgwb_2001_onwards.rename(columns=column_mapping)

In [ ]:
cgwb_2001_onwards_renamed

#**4. Displaying the available wells in CGWB data as points on India map**




In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import json

# Create a function to convert the .geo column to Shapely Point objects
def convert_geo_to_point(geo_str):
    geo_json = json.loads(geo_str)
    coordinates = geo_json['coordinates']
    return Point(coordinates)

# Create a function to convert the .geo column to Shapely Point objects
def convert_geo_to_point(geo_str):
    geo_json = json.loads(geo_str)
    coordinates = geo_json['coordinates']
    return Point(coordinates)

# Apply the conversion function to create a 'geometry' column
cgwb_2001_onwards_renamed['geometry'] = cgwb_2001_onwards_renamed['.geo'].apply(convert_geo_to_point)

# Create a GeoDataFrame
gdf = gpd.GeoDataFrame(cgwb_2001_onwards_renamed, geometry='geometry')

# Plot the GeoDataFrame
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, marker='o', color='blue', markersize=0.5, edgecolor='black', label='Well')

# Add title and axes labels
plt.title('Map of Well Locations mentioned in the CGWB data from 2001')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.legend()
plt.show()

In [ ]:
import numpy as np

#**3. Finding out the number of consecutive data available per well**





In [ ]:
import pandas as pd

# Columns to be removed
columns_to_remove_to_check_data_availability = ['objectid', 'pointInRegion', 'station_id', 'station_name', '.geo','geometry']

# Drop the specified columns
data = cgwb_2001_onwards_renamed.drop(columns=columns_to_remove_to_check_data_availability)

data

In [ ]:
import pandas as pd


# Function to calculate consecutive non-NaN pairs excluding the first four columns
def consecutive_non_nan_pairs(row):
    non_nan_pairs = 0
    consecutive_count = 0
    for value in row:
        if not pd.isna(value):
            consecutive_count += 1
            if consecutive_count == 2:  # Check if two consecutive non-NaN values are found
                non_nan_pairs += 1
                consecutive_count = 1  # Reset consecutive count to 1 to check for overlapping pairs
        else:
            consecutive_count = 0  # Reset consecutive count if NaN is encountered
    return non_nan_pairs

In [ ]:
# Apply the function to each row in the dataframe
cgwb_2001_onwards_with_consecutive_count = cgwb_2001_onwards_renamed.copy()

cgwb_2001_onwards_with_consecutive_count['consecutive_non_nan_pairs'] = data.apply(consecutive_non_nan_pairs, axis=1)

In [ ]:
cgwb_2001_onwards_with_consecutive_count

In [ ]:
# Create a new dataframe to store the results
results = cgwb_2001_onwards_with_consecutive_count[['station_id', 'consecutive_non_nan_pairs']]

# Display the results
print(results)

In [ ]:
results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Assuming 'cgwb_2001_onwards_renamed' is your dataframe
# The results dataframe already contains the necessary data
#results = filtered_above_MP_gdf[['station_id', 'consecutive_non_nan_pairs']]

# Calculate mean, median, and mode
mean_value = results['consecutive_non_nan_pairs'].mean()
median_value = results['consecutive_non_nan_pairs'].median()

print(f"Mean: {mean_value}")
print(f"Median: {median_value}")

# Calculate quartiles and percentiles
Q1 = results['consecutive_non_nan_pairs'].quantile(0.25)
Q3 = results['consecutive_non_nan_pairs'].quantile(0.75)
P10 = results['consecutive_non_nan_pairs'].quantile(0.10)
P20 = results['consecutive_non_nan_pairs'].quantile(0.20)

# Calculate the number of wells for each percentile and quartile
num_wells_P10 = results[results['consecutive_non_nan_pairs'] <= P10].shape[0]
num_wells_P20 = results[results['consecutive_non_nan_pairs'] <= P20].shape[0]
num_wells_Q1 = results[results['consecutive_non_nan_pairs'] <= Q1].shape[0]
num_wells_median = results[results['consecutive_non_nan_pairs'] <= median_value].shape[0]
num_wells_Q3 = results[results['consecutive_non_nan_pairs'] <= Q3].shape[0]

# Total number of wells
total_wells = results.shape[0]

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(results['consecutive_non_nan_pairs'], bins=range(0, max(results['consecutive_non_nan_pairs']) + 5, 5), edgecolor='black')
plt.title('Histogram of number of consecutive Non zero pair of values present in the CGWB data (2001-2023)')
plt.xlabel('Number of consecutive Non zero pair of values')
plt.ylabel('Number of Wells')
plt.xticks(range(0, max(results['consecutive_non_nan_pairs']) + 5, 5))
plt.grid(axis='y')
plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n',
                         horizontalalignment='right', fontsize=10)
plt.show()

# Plot inverted box plot
plt.figure(figsize=(10, 6))
sns.boxplot(y=results['consecutive_non_nan_pairs'], color='lightyellow')
plt.title('Box Plot of number of consecutive Non zero pair of values present in the CGWB data (2001-2023)')
plt.ylabel('Number of consecutive Non zero pair of values')

# Annotate the quartiles and percentiles
plt.text(0.8, P10, f'10th Percentile: {P10:.2f} ({num_wells_P10} wells)', horizontalalignment='center', verticalalignment='center', color='green')
plt.text(0.8, P20, f'20th Percentile: {P20:.2f} ({num_wells_P20} wells)', horizontalalignment='center', verticalalignment='center', color='green')
plt.text(0.8, Q1, f'Q1: {Q1:.2f} ({num_wells_Q1} wells)', horizontalalignment='center', verticalalignment='center', color='blue')
plt.text(0.8, median_value, f'Median: {median_value:.2f} ({num_wells_median} wells)', horizontalalignment='center', verticalalignment='center', color='red')
plt.text(0.8, Q3, f'Q3: {Q3:.2f} ({num_wells_Q3} wells)', horizontalalignment='center', verticalalignment='center', color='blue')

# Remove x-axis label as it's not used in the plot
plt.xlabel('')
plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n'
                         f'The number of wells in brackets represents the number of wells whose data has the number of consecutive non-zero pairs of entries less than or equal to the value represented at the point.',
                         horizontalalignment='right', fontsize=10)
plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
import pickle

In [ ]:
# Define the file path
file_path = f'{WORKSPACE_ROOT}/1. Analyzing CGWB data/output/cgwb_from_2001_preprocessed_data.pickle'

# Save the DataFrame as a pickle file
cgwb_2001_onwards_with_consecutive_count.to_pickle(file_path)

print(f"DataFrame saved successfully at {file_path}")